<a href="https://colab.research.google.com/github/2303a510c3/Quantum-Computing/blob/main/product_03_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install qiskit qiskit-aer numpy scipy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.6/9.6 MB 73.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.2/14.2 MB 99.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 61.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [qiskit-aer]


In [ ]:
"""
Hybrid Quantum Optimization System
Algorithm: QAOA
Problem: Max-Cut
Simulator: Qiskit Aer (ideal and noisy)
Optimizer: COBYLA
"""

import argparse
import csv
import json
import time
from pathlib import Path

import numpy as np
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from qiskit_aer.noise import (
    NoiseModel,
    ReadoutError,
    depolarizing_error,
)
from scipy.optimize import minimize


# -------------------------------
# 1. Define the optimization problem
# -------------------------------

NUM_QUBITS = 4

EDGES = [
    (0, 1),
    (1, 2),
    (2, 3),
    (3, 0),
    (0, 2),
]


def cut_value(bit_index):
    """Calculate the Max-Cut value of a bitstring."""

    value = 0

    for u, v in EDGES:
        bit_u = (bit_index >> u) & 1
        bit_v = (bit_index >> v) & 1

        if bit_u != bit_v:
            value += 1

    return value


def exact_max_cut():
    """Find the exact optimum using classical brute force."""

    scores = [
        cut_value(i)
        for i in range(2**NUM_QUBITS)
    ]

    optimum = max(scores)

    solutions = [
        format(i, f"0{NUM_QUBITS}b")
        for i, score in enumerate(scores)
        if score == optimum
    ]

    return optimum, solutions


# -------------------------------
# 2. Build the QAOA circuit
# -------------------------------

def build_qaoa_circuit(params, layers):
    """
    Construct the parameterized QAOA circuit.

    Each layer contains:
    - Cost unitary
    - Mixer unitary
    """

    if len(params) != 2 * layers:
        raise ValueError("Incorrect number of QAOA parameters.")

    circuit = QuantumCircuit(NUM_QUBITS)

    # Create an equal superposition
    circuit.h(range(NUM_QUBITS))

    for layer in range(layers):

        gamma = float(params[2 * layer])
        beta = float(params[2 * layer + 1])

        # Cost operator for Max-Cut
        for u, v in EDGES:
            circuit.rzz(-gamma, u, v)

        # Mixer operator
        for qubit in range(NUM_QUBITS):
            circuit.rx(2 * beta, qubit)

    circuit.measure_all()

    return circuit


# -------------------------------
# 3. Create the noisy simulator
# -------------------------------

def make_noisy_simulator(
    p_one_qubit=0.002,
    p_two_qubit=0.01,
    p_readout=0.02,
):
    """Create an illustrative quantum noise model."""

    noise_model = NoiseModel()

    # Single-qubit depolarizing error
    one_qubit_error = depolarizing_error(
        p_one_qubit, 1
    )

    # Two-qubit depolarizing error
    two_qubit_error = depolarizing_error(
        p_two_qubit, 2
    )

    # Measurement / readout error
    readout_error = ReadoutError([
        [1 - p_readout, p_readout],
        [p_readout, 1 - p_readout],
    ])

    noise_model.add_all_qubit_quantum_error(
        one_qubit_error,
        ["x", "sx"],
    )

    noise_model.add_all_qubit_quantum_error(
        two_qubit_error,
        ["cx"],
    )

    noise_model.add_all_qubit_readout_error(
        readout_error
    )

    return AerSimulator(noise_model=noise_model)


# -------------------------------
# 4. Execute the quantum circuit
# -------------------------------

def run_counts(circuit, backend, shots, seed):

    compiled = transpile(
        circuit,
        backend=backend,
        optimization_level=1,
    )

    start = time.perf_counter()

    result = backend.run(
        compiled,
        shots=shots,
        seed_simulator=seed,
    ).result()

    elapsed = time.perf_counter() - start

    return result.get_counts(), compiled, elapsed


# -------------------------------
# 5. Convert counts to probabilities
# -------------------------------

def counts_to_probabilities(counts):

    probabilities = np.zeros(
        2**NUM_QUBITS,
        dtype=float,
    )

    total_shots = sum(counts.values())

    if total_shots == 0:
        raise ValueError("No measurement results received.")

    for bitstring, count in counts.items():

        clean_bits = bitstring.replace(" ", "")

        index = int(clean_bits, 2)

        probabilities[index] += count / total_shots

    return probabilities


# -------------------------------
# 6. Calculate performance metrics
# -------------------------------

def calculate_metrics(probabilities, optimum):

    expected_cut = 0.0
    optimal_probability = 0.0

    for index, probability in enumerate(probabilities):

         score = cut_value(index)

         expected_cut += probability * score

         if score == optimum:
             optimal_probability += probability

    approximation_ratio = (
        expected_cut / optimum
        if optimum > 0
        else 0.0
    )

    return {
        "expected_cut": float(expected_cut),
        "optimal_solution_probability": float(
            optimal_probability
        ),
        "approximation_ratio": float(
            approximation_ratio
        ),
    }


def observed_metrics(counts, optimum, runtime):

    probabilities = counts_to_probabilities(counts)

    metrics = calculate_metrics(
        probabilities,
        optimum,
    )

    observed_scores = [
        cut_value(int(bits.replace(" ", ""), 2))
        for bits, count in counts.items()
        if count > 0
    ]

    metrics["best_cut_observed"] = max(observed_scores)

    metrics["execution_time_seconds"] = float(runtime)

    return metrics


# -------------------------------
# 7. Readout error mitigation
# -------------------------------

def estimate_readout_matrix(backend, shots, seed):
    """
    Calibrate each qubit using |0> and |1> states.

    Build a joint assignment matrix assuming
    independent readout errors.
    """

    zero_circuit = QuantumCircuit(NUM_QUBITS)
    zero_circuit.measure_all()

    one_circuit = QuantumCircuit(NUM_QUBITS)
    one_circuit.x(range(NUM_QUBITS))
    one_circuit.measure_all()

    circuits = transpile(
        [zero_circuit, one_circuit],
        backend=backend,
        optimization_level=1,
    )

    result = backend.run(
        circuits,
        shots=shots,
        seed_simulator=seed,
    ).result()

    zero_counts = result.get_counts(0)
    one_counts = result.get_counts(1)

    matrices = []

    for qubit in range(NUM_QUBITS):

        p_1_given_0 = sum(
            count
            for bits, count in zero_counts.items()
            if bits.replace(" ", "")[-1 - qubit] == "1"
        ) / shots

        p_0_given_1 = sum(
            count
            for bits, count in one_counts.items()
            if bits.replace(" ", "")[-1 - qubit] == "0"
        ) / shots

        matrix = np.array([
            [1 - p_1_given_0, p_0_given_1],
            [p_1_given_0, 1 - p_0_given_1],
        ])

        matrices.append(matrix)

    dimension = 2**NUM_QUBITS

    assignment = np.zeros(
        (dimension, dimension),
        dtype=float,
    )

    for measured in range(dimension):

        for actual in range(dimension):

            probability = 1.0

            for qubit in range(NUM_QUBITS):

                measured_bit = (measured >> qubit) & 1
                actual_bit = (actual >> qubit) & 1

                probability *= matrices[qubit][
                    measured_bit, actual_bit
                ]

            assignment[measured, actual] = probability

    return assignment


def mitigate_readout(counts, assignment):

    observed = counts_to_probabilities(counts)

    # Least-squares correction
    estimated = np.linalg.lstsq(
        assignment,
        observed,
        rcond=None,
    )[0]

    # Probability distributions cannot contain negatives
    estimated = np.clip(estimated, 0.0, None)

    total = estimated.sum()

    if total <= 0:
        return observed

    return estimated / total


# -------------------------------
# 8. Save results
# -------------------------------

def save_results(output_dir, report):

    output_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    json_path = output_dir / "results.json"

    with json_path.open("w", encoding="utf-8") as f:
        json.dump(report, f, indent=2)

    csv_path = output_dir / "comparison.csv"

    modes = [
        "ideal",
        "noisy",
        "noisy_readout_mitigated",
    ]

    with csv_path.open(
        "w",
        newline="",
        encoding="utf-8",
    ) as f:

        writer = csv.writer(f)

        writer.writerow([
            "execution_mode",
            "expected_cut",
            "approximation_ratio",
            "optimal_solution_probability",
        ])

        for mode in modes:

            metrics = report["metrics"][mode]

            writer.writerow([
                mode,
                metrics["expected_cut"],
                metrics["approximation_ratio"],
                metrics["optimal_solution_probability"],
            ])

    print("\nSaved:", json_path)
    print("Saved:", csv_path)


# -------------------------------
# 9. Main hybrid optimization
# -------------------------------

def main():

    parser = argparse.ArgumentParser(
        description="Hybrid QAOA Max-Cut optimization"
    )

    parser.add_argument("--layers", type=int, default=1)
    parser.add_argument("--maxiter", type=int, default=40)
    parser.add_argument(
        "--optimization-shots",
        type=int,
        default=512,
    )
    parser.add_argument("--shots", type=int, default=2048)
    parser.add_argument("--seed", type=int, default=7)
    parser.add_argument(
        "--output",
        type=Path,
        default=Path("outputs"),
    )

    # Use empty list args=[] to prevent argparse from checking sys.argv inside Jupyter notebooks
    args = parser.parse_args(args=[])

    if args.layers < 1:
        parser.error("--layers must be at least 1")

    if min(
        args.maxiter,
        args.optimization_shots,
        args.shots,
    ) < 1:
        parser.error("Iteration and shot counts must be positive.")

    optimum, optimal_solutions = exact_max_cut()

    ideal_backend = AerSimulator()

    noisy_backend = make_noisy_simulator()

    rng = np.random.default_rng(args.seed)

    initial_params = rng.uniform(
        0.0,
        np.pi,
        size=2 * args.layers,
    )

    evaluations = 0

    print("=" * 55)
    print(" HYBRID QUANTUM OPTIMIZATION SYSTEM")
    print("=" * 55)

    print("Algorithm: QAOA")
    print("Problem: Max-Cut")
    print("Number of qubits:", NUM_QUBITS)
    print("Number of edges:", len(EDGES))
    print("Exact optimal cut:", optimum)

    # Classical optimizer uses the noisy simulator
    def objective(params):

        nonlocal evaluations
        evaluations += 1

        circuit = build_qaoa_circuit(
            params,
            args.layers,
        )

        counts, _, _ = run_counts(
            circuit,
            noisy_backend,
            args.optimization_shots,
            args.seed + evaluations,
        )

        probabilities = counts_to_probabilities(counts)

        metrics = calculate_metrics(
            probabilities,
            optimum,
        )

        # Minimize the negative expected cut
        return -metrics["expected_cut"]

    print("\nOptimizing QAOA parameters...")

    optimization_start = time.perf_counter()

    result = minimize(
        objective,
        initial_params,
        method="COBYLA",
        options={
            "maxiter": args.maxiter,
            "rhobeg": 0.5,
            "tol": 1e-3,
        },
    )

    optimization_time = (
        time.perf_counter() - optimization_start
    )

    best_params = np.asarray(result.x, dtype=float)

    print("Optimization completed.")
    print("Optimized parameters:", best_params)

    final_circuit = build_qaoa_circuit(
        best_params,
        args.layers,
    )

    # Ideal execution
    ideal_counts, ideal_compiled, ideal_time = run_counts(
        final_circuit,
        ideal_backend,
        args.shots,
        args.seed + 10000,
    )

    # Noisy execution
    noisy_counts, noisy_compiled, noisy_time = run_counts(
        final_circuit,
        noisy_backend,
        args.shots,
        args.seed + 20000,
    )

    ideal_metrics = observed_metrics(
        ideal_counts,
        optimum,
        ideal_time,
    )

    noisy_metrics = observed_metrics(
        noisy_counts,
        optimum,
        noisy_time,
    )

    # Calibrate and mitigate readout errors
    assignment = estimate_readout_matrix(
        noisy_backend,
        args.shots,
        args.seed + 30000,
    )

    mitigated_probabilities = mitigate_readout(
        noisy_counts,
        assignment,
    )

    mitigated_metrics = calculate_metrics(
        mitigated_probabilities,
        optimum,
    )

    mitigated_metrics["best_cut_observed"] = (
        noisy_metrics["best_cut_observed"]
    )

    mitigated_metrics["execution_time_seconds"] = (
        noisy_time
    )

    report = {
        "project": "Hybrid Quantum Optimization System",
        "algorithm": "QAOA",

        "problem": {
            "name": "Max-Cut",
            "num_qubits": NUM_QUBITS,
            "edges": [list(edge) for edge in EDGES],
            "exact_optimum_cut": optimum,
            "optimal_bitstrings": optimal_solutions,
        },

        "settings": {
            "layers": args.layers,
            "shots": args.shots,
            "optimization_shots": args.optimization_shots,
            "optimizer": "COBYLA",
            "max_iterations": args.maxiter,
            "optimizer_evaluations": evaluations,
            "optimizer_success": bool(result.success),
            "optimizer_message": str(result.message),
            "optimization_time_seconds": optimization_time,
            "noise_model": {
                "one_qubit_error": 0.002,
                "two_qubit_error": 0.01,
                "readout_error": 0.02,
            },
            "readout_mitigation": "Least-squares assignment-matrix correction",
        },

        "optimized_parameters": best_params.tolist(),

        "circuit": {
            "ideal_depth": int(ideal_compiled.depth()),
            "ideal_operations": {
                str(k): int(v)
                for k, v in ideal_compiled.count_ops().items()
            },
            "noisy_depth": int(noisy_compiled.depth()),
            "noisy_operations": {
                str(k): int(v)
                for k, v in noisy_compiled.count_ops().items()
            },
        },

        "metrics": {
            "ideal": ideal_metrics,
            "noisy": noisy_metrics,
            "noisy_readout_mitigated": mitigated_metrics,
        },
    }

    save_results(args.output, report)

    print("\n" + "=" * 55)
    print(" PERFORMANCE COMPARISON")
    print("=" * 55)

    print("Exact optimal cut:", optimum)

    modes = [
        ("Ideal simulator", "ideal"),
        ("Noisy simulator", "noisy"),
        (
            "Noisy + readout mitigation",
            "noisy_readout_mitigated",
        ),
    ]

    for title, key in modes:

        metrics = report["metrics"][key]

        print(f"\n{title}")
        print(
            "Expected cut:",
            round(metrics["expected_cut"], 4),
        )
        print(
            "Approximation ratio:",
            round(metrics["approximation_ratio"], 4),
        )
        print(
            "Optimal-solution probability:",
            round(
                metrics["optimal_solution_probability"],
                4,
            ),
        )

    print("\nOptimizer evaluations:", evaluations)
    print(
        "Optimization time:",
        round(optimization_time, 3),
        "seconds",
    )

    print("\nProject execution completed successfully.")
    print("Check the outputs folder for JSON and CSV reports.")

    print(
        "\nNote: The noise model is illustrative and does not "
        "represent a calibrated quantum processor."
    )


if __name__ == "__main__":
    main()

 HYBRID QUANTUM OPTIMIZATION SYSTEM
Algorithm: QAOA
Problem: Max-Cut
Number of qubits: 4
Number of edges: 5
Exact optimal cut: 4

Optimizing QAOA parameters...
Optimization completed.
Optimized parameters: [2.13847356 2.72078768]

Saved: outputs/results.json
Saved: outputs/comparison.csv

 PERFORMANCE COMPARISON
Exact optimal cut: 4

Ideal simulator
Expected cut: 2.9077
Approximation ratio: 0.7269
Optimal-solution probability: 0.3604

Noisy simulator
Expected cut: 2.8882
Approximation ratio: 0.722
Optimal-solution probability: 0.3477

Noisy + readout mitigation
Expected cut: 2.924
Approximation ratio: 0.731
Optimal-solution probability: 0.3666

Optimizer evaluations: 19
Optimization time: 1.538 seconds

Project execution completed successfully.
Check the outputs folder for JSON and CSV reports.

Note: The noise model is illustrative and does not represent a calibrated quantum processor.
